In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# def analyze_audio_distribution(directory, num_files=3020):
#     files = glob.glob(os.path.join(directory, "*.wav"))[:num_files]
    
#     all_rms = []
#     all_zcr = [] 
    
#     for f in tqdm(files, desc=f"Analyzing {directory.split('/')[-1]}"):
#         y, sr = torchaudio.load(f)
#         y = y.numpy()
         
#         rms = np.sqrt(np.mean(y**2))
#         all_rms.append(rms)
         
#         zcr = np.mean(np.abs(np.diff(np.sign(y))))
#         all_zcr.append(zcr)
#     mean_rms = np.mean(all_rms)
#     std_rms = np.std(all_rms)
    
#     mean_zcr = np.mean(all_zcr)
#     std_zcr = np.std(all_zcr)
    
#     print(f"TEST SET LOUDNESS  Mean: {mean_rms:.5f} | Std Dev (Variance): {std_rms:.5f}")
#     print(f"TEST SET NOISINESS  Mean: {mean_zcr:.5f} | Std Dev (Variance): {std_zcr:.5f}")    
#     return np.mean(all_rms), np.mean(all_zcr)
 
# test_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'
# test_rms, test_zcr = analyze_audio_distribution(test_dir)
  

In [ ]:
# import librosa
# import numpy as np
# import glob
# import os
# from tqdm import tqdm

# def check_spectral_centroid(directory, num_files=3020):
#     files = glob.glob(os.path.join(directory, "*.wav"))[:num_files]
#     all_centroids = []
    
#     for f in tqdm(files, desc="Calculating Spectral Centroid"): 
#         y, sr = librosa.load(f, sr=22050, mono=True)
        
#         centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
#         all_centroids.append(np.mean(centroid))
        
#     mean_cent = np.mean(all_centroids)
#     std_cent = np.std(all_centroids)
#     return mean_cent, std_cent

# test_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'
# mean_c, std_c = check_spectral_centroid(test_dir)
# print(f" Mean Centroid: {mean_c } - Variance: {std_c } ")

In [ ]:
# !pip install --upgrade wandb==0.22.3

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")


In [ ]:
# import wandb
# wandb.login(key=secret_value_0)

In [ ]:
import kagglehub
import os, gc, glob, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time 
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import wandb
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models

from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings("ignore")


In [ ]:
import random
 
NOISE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
noise_files = [os.path.join(NOISE_DIR, f) for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]

In [ ]:
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
 
BASE_DIR     = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR    = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_DIR    = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master"
TEST_DIR     = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
TEST_CSV     = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
SAVE_DIR     = '/kaggle/working'
PRECOMP_TRAIN = '/kaggle/input/datasets/dhruvilsatasiya/pecomputed-dataset-t1dlgenai-train-10k-test-aug'
PRECOMP_VAL   = '/kaggle/input/datasets/dhruvilsatasiya/pecomputed-dataset-t1dlgenai-val-10k-test-aug'

SAMPLE_RATE  = 22050
CROP_SEC     = 6.0
CROP_LEN     = int(SAMPLE_RATE * CROP_SEC)
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
NUM_TRAIN     = 10000  
NUM_VAL      = 1500    
BATCH        = 32 
EPOCHS   = 14 

GENRES = sorted(["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"])
GENRE2ID = {g: i for i, g in enumerate(GENRES)}
ID2GENRE = {i: g for i, g in enumerate(GENRES)}

TARGET_RMS_MEAN = 0.14120
TARGET_RMS_STD  = 0.02697


In [ ]:
# def cache_all_stems(): 
#     stems_cache = {}
#     stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
    
#     for genre in GENRES:
#         genre_path = os.path.join(STEMS_DIR, genre)
#         if not os.path.exists(genre_path):
#             print(f"WARNING: {genre_path} not found"); continue
        
#         songs = sorted([d for d in os.listdir(genre_path)
#                         if os.path.isdir(os.path.join(genre_path, d))])
#         stems_cache[genre] = {}
        
#         for song in tqdm(songs, desc=f"Caching {genre}", leave=False):
#             song_stems = {}
#             song_dir = os.path.join(genre_path, song)
#             valid = True
#             for sn in stem_names:
#                 sp = os.path.join(song_dir, sn)
#                 if not os.path.exists(sp):
#                     valid = False; break
#                 try:
#                     w, sr = torchaudio.load(sp)
#                     w = w.mean(dim=0, keepdim=True)  # mono
#                     if sr != SAMPLE_RATE:
#                         w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
#                     song_stems[sn] = w
#                 except Exception as e:
#                     print(f"  Skip {sp}: {e}"); valid = False; break
#             if valid:
#                 stems_cache[genre][song] = song_stems
    
#     total = sum(len(v) for v in stems_cache.values())
#     print(f" Cached {total} songs across {len(stems_cache)} genres")
#     return stems_cache


# def cache_noise(): 
#     noise_cache = []
#     noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav'))
#     for nf in tqdm(noise_files, desc="Caching noise", leave=False):
#         try:
#             w, sr = torchaudio.load(nf)
#             w = w.mean(dim=0, keepdim=True)
#             if sr != SAMPLE_RATE:
#                 w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
#             noise_cache.append(w)
#         except:
#             pass
#     print(f"Cached {len(noise_cache)} noise clips")
#     return noise_cache

 
# stems_cache = cache_all_stems()
# noise_cache = cache_noise()
 
# train_songs = {}
# val_songs = {}
# for genre, songs_dict in stems_cache.items():
#     song_ids = sorted(songs_dict.keys())
#     split = int(0.8 * len(song_ids))
#     train_songs[genre] = {s: stems_cache[genre][s] for s in song_ids[:split]}
#     val_songs[genre]   = {s: stems_cache[genre][s] for s in song_ids[split:]}




In [ ]:
# import shutil
# import os

# # Define the path to the directory you want to delete
# directory_path = "/kaggle/working/precomputed_train"

# # Check if the directory exists before attempting to delete
# if os.path.exists(directory_path):
#     shutil.rmtree(directory_path)
#     print(f"Directory '{directory_path}' and all its contents have been removed.")
# else:
#     print(f"Directory '{directory_path}' not found.")


In [ ]:

# mel_transform = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
# db_transform = T.AmplitudeToDB()


# def crop_or_pad(wave, length):
#     if wave.shape[1] >= length:
#         start = random.randint(0, wave.shape[1] - length)
#         return wave[:, start:start + length]
#     return F.pad(wave, (0, length - wave.shape[1]))


# def generate_one_mashup(songs_cache, noise_cache, is_train=True): 
#     genre = random.choice(GENRES)
#     label = GENRE2ID[genre]
    
#     available = list(songs_cache[genre].keys())
#     if len(available) < 2:
#         available = available * 4
    
#     stem_names = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
#     chosen = random.choices(available, k=4)
#     buffer_len = int(CROP_LEN * 1.2)
    
#     mixed = torch.zeros(1, CROP_LEN)
#     for i, sn in enumerate(stem_names):
#         wave = songs_cache[genre][chosen[i]][sn]
         
#         wave = crop_or_pad(wave, buffer_len)
         
#         if is_train and random.random() < 0.5:
#             factor = random.uniform(0.85, 1.15)
#             new_len = int(wave.shape[1] / factor)  
#             wave = F.interpolate(wave.unsqueeze(0), size=new_len, mode='linear',
#                                  align_corners=False).squeeze(0)
        
#         wave = crop_or_pad(wave, CROP_LEN)
         
#         if is_train and random.random() < 0.15:
#             continue
        
#         vol = random.uniform(0.5, 1.3) if is_train else 1.0
#         mixed = mixed + wave * vol
     
#     add_noise = (is_train and random.random() < 0.85) or \
#                 (not is_train and random.random() < 0.7)
#     if add_noise and noise_cache:
#         n_clips = random.randint(1, 3) if is_train else 1
#         for _ in range(n_clips):
#             nw = crop_or_pad(random.choice(noise_cache), CROP_LEN)
#             snr = random.uniform(0.05, 0.8) if is_train else random.uniform(0.1, 0.4)
#             mixed = mixed + nw * snr
     
#     target_rms = max(0.01, random.gauss(TARGET_RMS_MEAN, TARGET_RMS_STD))
#     cur_rms = torch.sqrt(torch.mean(mixed ** 2))
#     if cur_rms > 1e-6:
#         mixed = mixed * (target_rms / cur_rms)
#     mixed = torch.clamp(mixed, -1.0, 1.0)
     
#     with torch.no_grad():
#         mel = mel_transform(mixed)
#         mel_db = db_transform(mel)
#         mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    
#     return mel_db, label


# def precompute_dataset(songs_cache, noise_cache, output_dir, num_samples, is_train=True): 
#     os.makedirs(output_dir, exist_ok=True)
    
#     existing = len(glob.glob(os.path.join(output_dir, '*.pt'))) 
    
#     for i in tqdm(range(num_samples), desc=f"pre compute srtart  {output_dir.split('/')[-1]}"):
#         mel, label = generate_one_mashup(songs_cache, noise_cache, is_train)
#         torch.save({'spec': mel, 'label': torch.tensor(label, dtype=torch.long)},
#                    os.path.join(output_dir, f'{i:05d}.pt'))

 
# precompute_dataset(train_songs, noise_cache, PRECOMP_TRAIN, NUM_TRAIN, is_train=True)
# precompute_dataset(val_songs, noise_cache, PRECOMP_VAL, NUM_VAL, is_train=False) 

In [ ]:
# import kagglehub
# import os

# # Define your dataset handle and local directory path
# # Replace '<KAGGLE_USERNAME>' with your actual Kaggle username
# # Replace '<DATASET_SLUG>' with your desired name for the dataset URL
# handle = 'dhruvilsatasiya/pecomputed-dataset-t1dlgenai-val-10k-test-aug'
# local_dataset_dir = '/kaggle/working/precomp_val'

# # Optional: Add version notes
# version_notes = 'Upload of data generated in notebook session'

# # Upload the dataset
# # Note: This will create a new dataset or a new version of an existing one
# # Make sure you have the necessary permissions (e.g., you own the handle's namespace)
# print(f"Uploading data to KaggleHub dataset: {handle} from directory: {local_dataset_dir}")
# kagglehub.dataset_upload(handle, local_dataset_dir, version_notes=version_notes)

# print("Dataset upload complete.")


In [ ]:
class PrecomputedDataset(Dataset): 
    
    def __init__(self, directory, is_train=True):
        self.files = sorted(glob.glob(os.path.join(directory, '*.pt')))
        self.is_train = is_train
        self.time_mask = T.TimeMasking(time_mask_param=40)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=20)
        assert len(self.files) > 0, f"No .pt files found in {directory}"
        print(f"  Dataset: {len(self.files)} files from {directory}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(self.files[idx], weights_only=True)
        spec = data['spec']
        label = data['label']
         
        if self.is_train:
            if random.random() < 0.5: spec = self.time_mask(spec)
            if random.random() < 0.5: spec = self.freq_mask(spec)
        
        return spec, label

In [ ]:
class SEBlock(nn.Module): 
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch // r), nn.ReLU(),
            nn.Linear(ch // r, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)


In [ ]:
class CRNN(nn.Module): 
    def __init__(self, num_classes=10):
        super().__init__()
        
        def conv_block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                SEBlock(cout),
                nn.MaxPool2d(2))
        
        self.cnn = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
        )
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.gru = nn.GRU(256, 128, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.3)
         
        self.attn = nn.Sequential(nn.Linear(256, 64), nn.Tanh(), nn.Linear(64, 1))
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.cnn(x)                        
        x = self.freq_pool(x).squeeze(2)      
        x = x.permute(0, 2, 1)                  
        x, _ = self.gru(x)                      
        w = torch.softmax(self.attn(x), dim=1)  
        x = (x * w).sum(dim=1)                 
        return self.fc(x)



In [ ]:
 
class PretrainedEfficientNet(nn.Module):
    def __init__(self, num_classes=10):
        super(PretrainedEfficientNet, self).__init__()
         
        self.efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
         
        original_conv = self.efficientnet.features[0][0]
        self.efficientnet.features[0][0] = nn.Conv2d(
            1, original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
         
        num_ftrs = self.efficientnet.classifier[1].in_features
        self.efficientnet.classifier[1] = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.efficientnet(x)


In [ ]:
# wandb.init(
#     # Set the wandb entity where your project will be logged (generally your team name).
#     entity="23f2004889-dl-genai-project",
#     # Set the wandb project where this run will be logged.
#     project="dl-genai-t126",
#     # Track hyperparameters and run metadata.
#     name="PretrainedEfficientNet-10k-test-aug",
#     config={ 
#         "architecture": "PretrainedEfficientNet-10k-test-aug", 
#         "epochs":14,
#     },
# )

In [ ]:

class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, pred, target):
        log_prob = F.log_softmax(pred, dim=-1)
        nll = -log_prob.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth = -log_prob.mean(dim=-1)
        return ((1 - self.smoothing) * nll + self.smoothing * smooth).mean()


In [ ]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def train_one_epoch(model, loader, optimizer, criterion, scaler, device, use_mixup=True, grad_accum=1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()
    
    for i, (x, y) in enumerate(tqdm(loader, leave=False)):
        x, y = x.to(device), y.to(device)
        
        if use_mixup and random.random() < 0.5:
            x, ya, yb, lam = mixup_data(x, y)
            with autocast():
                logits = model(x)
                loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
        else:
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
        
        loss = loss / grad_accum
        scaler.scale(loss).backward()
        
        if (i + 1) % grad_accum == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * grad_accum
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / len(loader), correct / total




In [ ]:

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())
    return f1_score(all_labels, all_preds, average='macro'),accuracy_score(all_labels, all_preds)


In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, device, grad_accum=1, use_wandb=False):
    criterion = LabelSmoothingCE(smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()
    best_f1 = 0.0
    
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, train_loader, optimizer, criterion,
                                     scaler, device, use_mixup=True, grad_accum=grad_accum)
        scheduler.step()
        f1, val_acc = validate(model, val_loader, device)
        lr = scheduler.get_last_lr()[0]
        
        print(f"Ep {epoch+1}/{epochs} - Loss: {loss:.4f}  Acc: {acc:.3f} - ValF1: {f1:.4f} - ValAcc: {val_acc:.3f} - LR: {lr:.6f}")
         
        wandb.log({
            "train_loss": loss, 
            "val_macro_f1": f1,
            "accuracy": acc,
            
            "learning_rate": lr
        })
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_effnet.pth'))
            print(f"best F1: {best_f1:.4f}")
    
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_effnet.pth'), map_location=device)) 
    return model, best_f1

In [ ]:
# train_ds = PrecomputedDataset(PRECOMP_TRAIN, is_train=True)
# val_ds   = PrecomputedDataset(PRECOMP_VAL, is_train=False)

# train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

# val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)  

# model_effnet = PretrainedEfficientNet(10).to(DEVICE)
# model_effnet, f1_effnet = train_model(model_effnet, train_loader, val_loader,epochs=EPOCHS, lr=1e-3, device=DEVICE )
# wandb.finish()

In [ ]:
# import kagglehub
# import os
# import shutil
 
# model_upload_dir = '/kaggle/working/PretrainedEfficientNet-10k-test-aug'
# os.makedirs(model_upload_dir, exist_ok=True)

# shutil.move('/kaggle/working/best_effnet.pth', f'{model_upload_dir}/best_effnet.pth')

# handle = 'dhruvilsatasiya/PretrainedEfficientNet-10k-test-aug/pytorch/default'
# kagglehub.model_upload(handle, model_upload_dir)


In [ ]:
test_df = pd.read_csv(TEST_CSV, dtype={'id': str})

mel_spec_inf = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
db_scale_inf = T.AmplitudeToDB()


def load_test_waveform(path):
    w, sr = torchaudio.load(path)
    w = w.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
    return w


def get_tta_crops(waveform, crop_len):
    crops = []
    total = waveform.shape[1]
    if total <= crop_len:
        crops.append(F.pad(waveform, (0, crop_len - total)))
    else:
        crops.append(waveform[:, :crop_len])
        mid = (total - crop_len) // 2
        crops.append(waveform[:, mid:mid + crop_len])
        crops.append(waveform[:, -crop_len:])
    return crops



In [ ]:
def mel_from_wave(wave):
    mel = mel_spec_inf(wave)
    mel = db_scale_inf(mel)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

    
download_path = kagglehub.model_download("dhruvilsatasiya/PretrainedEfficientNet-10k-test-aug/pytorch/default")
model_effnet = PretrainedEfficientNet(num_classes=10).to(DEVICE)
state_dict_path = os.path.join(download_path, 'best_effnet.pth')
model_effnet.load_state_dict(torch.load(state_dict_path)) 
model_effnet.eval()
results = []
with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
        raw_id = str(row['id'])
        
         
        fname = f"song{raw_id.zfill(4)}.wav"
        
        path = os.path.join(TEST_DIR, fname)
        if not os.path.exists(path):
            path = os.path.join(TEST_DIR, f"{raw_id.zfill(4)}.wav")
        
        wave = load_test_waveform(path)
        crops = get_tta_crops(wave, CROP_LEN)
        
        all_probs = torch.zeros(10, device=DEVICE)
        for crop in crops:
            mel = mel_from_wave(crop).unsqueeze(0).to(DEVICE)
            with autocast():
                logits_effnet = model_effnet(mel) 
            probs_effnet = torch.softmax(logits_effnet, dim=1).squeeze(0) 
            
            all_probs += probs_effnet  
        
        all_probs /= len(crops)
        results.append({'id': raw_id.zfill(4), 'genre': ID2GENRE[all_probs.argmax().item()]})
 

In [ ]:
submission = pd.DataFrame(results)[['id', 'genre']]
submission.to_csv('submission.csv', index=False)
 
print(submission.head(10))


In [ ]:
# !pip install panns-inference